In [ ]:
# Universal Setup
!pip install -q faiss-cpu chromadb qdrant-client rank-bm25 \
    sentence-transformers psutil pymupdf tiktoken groq google-genai \
    pandas matplotlib seaborn scikit-learn

import os, re, json, time, gc, random, glob
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

BASE    = '/content/drive/MyDrive/rag_term_paper'
PDF_DIR = f'{BASE}/data/raw_pdfs'
PROC    = f'{BASE}/data/processed'
EMB     = f'{BASE}/data/embeddings'
RES     = f'{BASE}/results'
for d in (PDF_DIR, PROC, EMB, RES):
    os.makedirs(d, exist_ok=True)
print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.


In [ ]:
# Load Frozen Assets
chunks    = json.load(open(f'{PROC}/chunks.json'))
queries   = json.load(open(f'{PROC}/eval_queries.json'))
chunk_ids = json.load(open(f'{EMB}/chunk_ids.json'))
doc_emb   = np.load(f'{EMB}/doc_embeddings.npy').astype('float32')
q_emb     = np.load(f'{EMB}/query_embeddings.npy').astype('float32')

chunk_text_by_id = {c['chunk_id']: c['text'] for c in chunks}
qidx = {q['query_id']: i for i, q in enumerate(queries)}
num_chunks, dim = doc_emb.shape
answerable_qids = [q['query_id'] for q in queries if q['answerable']]
print(f"Loaded: {num_chunks} chunks ({dim}-dim), {len(queries)} queries ({len(answerable_qids)} answerable)")

Loaded: 345 chunks (384-dim), 20 queries (15 answerable)


In [ ]:
# Shared Retrieval Functions (4 Systems)
import faiss
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def tok(t):
    return [w for w in re.findall(r'[a-z0-9]+', t.lower())
            if w not in ENGLISH_STOP_WORDS]

bm25 = BM25Okapi([tok(c['text']) for c in chunks])

flat = faiss.IndexFlatIP(dim)
flat.add(doc_emb)

hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 200
hnsw.hnsw.efSearch = 64
hnsw.add(doc_emb)

def dense_rank(qi, k=20):
    _, ix = flat.search(q_emb[qi:qi+1], k)
    return [chunk_ids[i] for i in ix[0]]

def dense_rank_hnsw(qi, k=20):
    _, ix = hnsw.search(q_emb[qi:qi+1], k)
    return [chunk_ids[i] for i in ix[0]]

def bm25_rank(text, k=20):
    sc = bm25.get_scores(tok(text))
    return [chunk_ids[i] for i in np.argsort(sc)[::-1][:k]]

def rrf(lists, k=60, top=20, weights=None):
    weights = weights or [1.0] * len(lists)
    score = {}
    for w, lst in zip(weights, lists):
        for r, cid in enumerate(lst, 1):
            score[cid] = score.get(cid, 0) + w / (k + r)
    return [c for c, _ in sorted(score.items(), key=lambda x: -x[1])[:top]]

def hybrid_rank(qi, top=20):
    return rrf([dense_rank(qi, 20),
                bm25_rank(queries[qi]['query'], 20)], top=top)

print("All 4 retrieval systems ready (Dense-Flat, Dense-HNSW, BM25, Hybrid-RRF)")

All 4 retrieval systems ready (Dense-Flat, Dense-HNSW, BM25, Hybrid-RRF)


In [ ]:
# 1- Layer 1: Vector Backend Benchmark
import psutil, chromadb
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

proc = psutil.Process()
rss = lambda: proc.memory_info().rss / 1024**2

def build_flat(X):
    ix = faiss.IndexFlatIP(X.shape[1]); ix.add(X)
    return lambda q, k: [int(i) for i in ix.search(q[None], k)[1][0]]

def build_hnsw(X, ef=64):
    ix = faiss.IndexHNSWFlat(X.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
    ix.hnsw.efConstruction = 200; ix.hnsw.efSearch = ef
    ix.add(X)
    return lambda q, k: [int(i) for i in ix.search(q[None], k)[1][0]]

def build_chroma(X):
    cl = chromadb.EphemeralClient()
    col = cl.create_collection(
        f'c{time.time_ns()}', metadata={'hnsw:space': 'cosine'})
    ids = [str(i) for i in range(len(X))]
    for s in range(0, len(X), 5000):
        col.add(ids=ids[s:s+5000], embeddings=X[s:s+5000].tolist())
    return lambda q, k: [int(i) for i in
        col.query(query_embeddings=[q.tolist()], n_results=k)['ids'][0]]

def build_qdrant(X):
    cl = QdrantClient(':memory:')
    cl.create_collection('c', vectors_config=VectorParams(
        size=X.shape[1], distance=Distance.COSINE))
    for s in range(0, len(X), 2000):
        e = min(s+2000, len(X))
        cl.upsert('c', points=[
            PointStruct(id=i, vector=X[i].tolist()) for i in range(s, e)])
    return lambda q, k: [p.id for p in
        cl.query_points('c', query=q.tolist(), limit=k).points]

BACKENDS = {'FAISS-Flat': build_flat, 'FAISS-HNSW': build_hnsw,
            'ChromaDB': build_chroma, 'Qdrant(local)': build_qdrant}

def run_backend(name, X, Q, truth, reps=20, warmup=5, k=10):
    gc.collect()
    r0 = rss(); t0 = time.perf_counter()
    search = BACKENDS[name](X)
    build_s = time.perf_counter() - t0
    ram = max(0.0, rss() - r0)
    for q in Q[:warmup]:
        search(q, k)
    all_ms, found = [], []
    for q in Q:
        for _ in range(reps if len(X) <= 1000 else 10):
            t = time.perf_counter()
            res = search(q, k)
            all_ms.append((time.perf_counter() - t) * 1e3)
        found.append(res)
    overlap = np.mean([len(set(f) & set(t)) / k
                       for f, t in zip(found, truth)])
    return {'Backend': name, 'N': len(X), 'Build_s': round(build_s, 4),
            'Vectors_per_s': round(len(X)/build_s, 1),
            'RSS_delta_MB': round(ram, 2),
            'Median_ms': round(np.median(all_ms), 3),
            'P95_ms': round(np.percentile(all_ms, 95), 3),
            'ANN_overlap@10': round(overlap, 4)}

# Run on real corpus (N=345)
truth_real = np.argsort(-(q_emb @ doc_emb.T), axis=1)[:, :10]
results = []
for name in BACKENDS:
    r = run_backend(name, doc_emb, q_emb, truth_real)
    results.append(r)
    print(r)

# Scaling to 10k synthetic
def synth(X, n, noise=0.05):
    base = X[np.random.randint(0, len(X), n)]
    Y = base + noise * np.random.randn(*base.shape).astype('float32')
    return (Y / np.linalg.norm(Y, axis=1, keepdims=True)).astype('float32')

X10k = synth(doc_emb, 10_000)
truth_10k = np.argsort(-(q_emb @ X10k.T), axis=1)[:, :10]
for name in BACKENDS:
    r = run_backend(name, X10k, q_emb, truth_10k)
    results.append(r)
    print(r)

df1 = pd.DataFrame(results)
df1.to_csv(f'{RES}/layer1_scaling_raw.csv', index=False)
print("\n=== LAYER 1 RESULTS ===")
display(df1)

{'Backend': 'FAISS-Flat', 'N': 345, 'Build_s': 0.0005, 'Vectors_per_s': 679777.5, 'RSS_delta_MB': 0.5, 'Median_ms': np.float64(0.022), 'P95_ms': np.float64(0.033), 'ANN_overlap@10': np.float64(1.0)}
{'Backend': 'FAISS-HNSW', 'N': 345, 'Build_s': 0.0322, 'Vectors_per_s': 10706.8, 'RSS_delta_MB': 0.09, 'Median_ms': np.float64(0.058), 'P95_ms': np.float64(0.09), 'ANN_overlap@10': np.float64(1.0)}
{'Backend': 'ChromaDB', 'N': 345, 'Build_s': 0.1836, 'Vectors_per_s': 1879.6, 'RSS_delta_MB': 28.92, 'Median_ms': np.float64(1.327), 'P95_ms': np.float64(1.787), 'ANN_overlap@10': np.float64(1.0)}
{'Backend': 'Qdrant(local)', 'N': 345, 'Build_s': 0.076, 'Vectors_per_s': 4537.0, 'RSS_delta_MB': 2.3, 'Median_ms': np.float64(1.207), 'P95_ms': np.float64(1.449), 'ANN_overlap@10': np.float64(1.0)}
{'Backend': 'FAISS-Flat', 'N': 10000, 'Build_s': 0.0053, 'Vectors_per_s': 1887279.9, 'RSS_delta_MB': 0.0, 'Median_ms': np.float64(6.064), 'P95_ms': np.float64(9.986), 'ANN_overlap@10': np.float64(1.0)}
{'Bac

,Backend,N,Build_s,Vectors_per_s,RSS_delta_MB,Median_ms,P95_ms,ANN_overlap@10
0,FAISS-Flat,345,0.0005,679777.5,0.50,0.022,0.033,1.000
1,FAISS-HNSW,345,0.0322,10706.8,0.09,0.058,0.090,1.000
2,ChromaDB,345,0.1836,1879.6,28.92,1.327,1.787,1.000
3,Qdrant(local),345,0.0760,4537.0,2.30,1.207,1.449,1.000
4,FAISS-Flat,10000,0.0053,1887279.9,0.00,6.064,9.986,1.000
5,FAISS-HNSW,10000,4.3798,2283.2,0.04,0.231,0.301,0.960
6,ChromaDB,10000,3.9464,2534.0,84.05,1.621,1.859,0.945
7,Qdrant(local),10000,1.0730,9319.3,70.05,15.539,32.844,1.000


In [ ]:
# 2 - HNSW efSearch Sweep (N=25,000)
X25k = synth(doc_emb, 25_000)
truth_25k = np.argsort(-(q_emb @ X25k.T), axis=1)[:, :10]
sweep = []
try:
    for ef in (32, 64, 128):
        BACKENDS['_tmp'] = lambda X, ef=ef: build_hnsw(X, ef)
        r = run_backend('_tmp', X25k, q_emb, truth_25k)
        r['Backend'] = f'FAISS-HNSW ef={ef}'; r['efSearch'] = ef
        sweep.append(r); print(r)
finally:
    BACKENDS.pop('_tmp', None)

df_sweep = pd.DataFrame(sweep)
df_sweep.to_csv(f'{RES}/layer1_hnsw_ef_sweep.csv', index=False)
print("\n=== HNSW SWEEP RESULTS ===")
display(df_sweep)

{'Backend': 'FAISS-HNSW ef=32', 'N': 25000, 'Build_s': 16.2375, 'Vectors_per_s': 1539.6, 'RSS_delta_MB': 36.64, 'Median_ms': np.float64(0.212), 'P95_ms': np.float64(0.292), 'ANN_overlap@10': np.float64(0.87), 'efSearch': 32}
{'Backend': 'FAISS-HNSW ef=64', 'N': 25000, 'Build_s': 16.3193, 'Vectors_per_s': 1531.9, 'RSS_delta_MB': 36.62, 'Median_ms': np.float64(0.34), 'P95_ms': np.float64(0.466), 'ANN_overlap@10': np.float64(0.955), 'efSearch': 64}
{'Backend': 'FAISS-HNSW ef=128', 'N': 25000, 'Build_s': 17.2391, 'Vectors_per_s': 1450.2, 'RSS_delta_MB': 36.62, 'Median_ms': np.float64(0.572), 'P95_ms': np.float64(0.808), 'ANN_overlap@10': np.float64(0.97), 'efSearch': 128}

=== HNSW SWEEP RESULTS ===


,Backend,N,Build_s,Vectors_per_s,RSS_delta_MB,Median_ms,P95_ms,ANN_overlap@10,efSearch
0,FAISS-HNSW ef=32,25000,16.2375,1539.6,36.64,0.212,0.292,0.870,32
1,FAISS-HNSW ef=64,25000,16.3193,1531.9,36.62,0.340,0.466,0.955,64
2,FAISS-HNSW ef=128,25000,17.2391,1450.2,36.62,0.572,0.808,0.970,128


In [ ]:
# Export Labelling Pool
rows = []
for qi, q in enumerate(queries):
    if not q['answerable']:
        continue
    d = dense_rank(qi, 10)
    b = bm25_rank(q['query'], 10)
    h = hybrid_rank(qi, 10)
    pool = list(dict.fromkeys(d[:5] + b[:5] + h[:5]))
    random.shuffle(pool)
    for cid in pool:
        rows.append({
            'query_id': q['query_id'],
            'query': q['query'],
            'chunk_id': cid,
            'text': chunk_text_by_id[cid][:600].replace('\n', ' '),
            'label': ''
        })

pd.DataFrame(rows).to_csv(f'{PROC}/qrels_to_label.csv', index=False)
print(f"{len(rows)} rows to label (for {len(answerable_qids)} queries)")

123 rows to label (for 15 queries)


In [ ]:
# 1 LLM-Assisted Pre-Labelling via Groq
import getpass
from groq import Groq

groq_key = getpass.getpass("Groq API key: ").strip()
groq_client = Groq(api_key=groq_key)

# Prefer strong non-Llama-3.3 models so pre-labeller != Phase 4 generator
PREFERRED = [
    'openai/gpt-oss-120b',
    'qwen/qwen3-32b',
    'meta-llama/llama-4-scout-17b-16e-instruct',
    'openai/gpt-oss-20b',
    'llama-3.1-8b-instant',
]
available = {m.id for m in groq_client.models.list().data}
LABEL_MODEL = next((m for m in PREFERRED if m in available), None)
assert LABEL_MODEL is not None, f"No suitable model found. Available: {sorted(available)}"
print(f"Using pre-labelling model: {LABEL_MODEL}")

pool_df = pd.read_csv(f'{PROC}/qrels_to_label.csv')

LABEL_PROMPT = """Rate how relevant this text chunk is for answering the question.
2 = Contains the answer or key information needed
1 = Related/on-topic but doesn't contain the answer
0 = Not relevant

Question: {query}
Chunk: {text}

Reply with ONLY the single digit: 0, 1, or 2"""

def label_with_groq(query, text, max_retries=6):
    prompt = LABEL_PROMPT.format(query=query, text=text)
    for attempt in range(max_retries):
        try:
            resp = groq_client.chat.completions.create(
                model=LABEL_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=16,
            )
            out = (resp.choices[0].message.content or "").strip()
            out = re.sub(r'<think>.*?</think>', '', out, flags=re.DOTALL).strip()
            m = re.search(r'[012]', out)
            return int(m.group()) if m else 0
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate_limit' in err.lower() or '503' in err:
                m = re.search(r'try again in ([\d.]+)s', err)
                wait = float(m.group(1)) + 1.5 if m else min(5 * (2 ** attempt), 60)
                print(f"  Rate-limited, sleeping {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"  Error on attempt {attempt+1}: {err[:120]}")
                time.sleep(2)
    raise RuntimeError("Exceeded max retries on Groq.")

labels = []
for i, row in pool_df.iterrows():
    lbl = label_with_groq(row['query'], row['text'])
    labels.append(lbl)
    if (i + 1) % 20 == 0 or (i + 1) == len(pool_df):
        print(f"  Labelled {i+1}/{len(pool_df)}...")
    time.sleep(2.1)

pool_df['label'] = labels
pool_df.to_csv(f'{PROC}/qrels_prelabeled.csv', index=False)
print(f"\nPre-labelled {len(pool_df)} rows. OPEN IN SHEETS AND VERIFY!")
print("Label-2 chunks per query:")
print(pool_df[pool_df.label == 2].groupby('query_id').size())

Groq API key: ··········
Using pre-labelling model: openai/gpt-oss-120b
  Labelled 20/123...
  Labelled 40/123...
  Labelled 60/123...
  Labelled 80/123...
  Labelled 100/123...
  Labelled 120/123...
  Labelled 123/123...

Pre-labelled 123 rows. OPEN IN SHEETS AND VERIFY!
Label-2 chunks per query:
Series([], dtype: int64)


In [ ]:
# 2 Populate Verified Pre-Labels into qrels_prelabeled.csv
PRE_LABELS = {
    ('q01', 'doc01_rag_original_chunk_0013'): 2,
    ('q01', 'doc01_rag_original_chunk_0002'): 2,
    ('q01', 'doc01_rag_original_chunk_0001'): 1,
    ('q01', 'doc01_rag_original_chunk_0003'): 1,
    ('q01', 'doc01_rag_original_chunk_0000'): 1,
    ('q01', 'doc01_rag_original_chunk_0019'): 1,
    ('q01', 'doc01_rag_original_chunk_0027'): 1,
    ('q01', 'doc02_rag_survey_chunk_0000'): 1,
    ('q02', 'doc02_rag_survey_chunk_0004'): 2,
    ('q02', 'doc02_rag_survey_chunk_0005'): 2,
    ('q02', 'doc02_rag_survey_chunk_0006'): 2,
    ('q02', 'doc02_rag_survey_chunk_0007'): 2,
    ('q02', 'doc02_rag_survey_chunk_0002'): 1,
    ('q02', 'doc02_rag_survey_chunk_0003'): 1,
    ('q02', 'doc09_rag_survey_huang_chunk_0044'): 0,
    ('q03', 'doc03_dpr_chunk_0002'): 2,
    ('q03', 'doc03_dpr_chunk_0000'): 1,
    ('q03', 'doc03_dpr_chunk_0001'): 1,
    ('q03', 'doc03_dpr_chunk_0003'): 1,
    ('q03', 'doc03_dpr_chunk_0017'): 1,
    ('q03', 'doc10_comprehensive_rag_chunk_0005'): 1,
    ('q04', 'doc05_self_rag_chunk_0003'): 2,
    ('q04', 'doc05_self_rag_chunk_0004'): 2,
    ('q04', 'doc02_rag_survey_chunk_0026'): 2,
    ('q04', 'doc05_self_rag_chunk_0000'): 1,
    ('q04', 'doc05_self_rag_chunk_0002'): 1,
    ('q04', 'doc05_self_rag_chunk_0018'): 1,
    ('q05', 'doc06_transformer_chunk_0005'): 2,
    ('q05', 'doc06_transformer_chunk_0000'): 1,
    ('q05', 'doc06_transformer_chunk_0001'): 1,
    ('q05', 'doc06_transformer_chunk_0002'): 1,
    ('q05', 'doc06_transformer_chunk_0003'): 1,
    ('q05', 'doc06_transformer_chunk_0004'): 1,
    ('q06', 'doc07_lora_chunk_0001'): 2,
    ('q06', 'doc07_lora_chunk_0006'): 2,
    ('q06', 'doc07_lora_chunk_0000'): 1,
    ('q06', 'doc07_lora_chunk_0014'): 1,
    ('q06', 'doc07_lora_chunk_0015'): 1,
    ('q07', 'doc08_flash_attention_chunk_0002'): 2,
    ('q07', 'doc08_flash_attention_chunk_0003'): 2,
    ('q07', 'doc08_flash_attention_chunk_0013'): 2,
    ('q07', 'doc08_flash_attention_chunk_0000'): 1,
    ('q07', 'doc08_flash_attention_chunk_0005'): 1,
    ('q07', 'doc08_flash_attention_chunk_0012'): 1,
    ('q07', 'doc08_flash_attention_chunk_0023'): 1,
    ('q07', 'doc08_flash_attention_chunk_0051'): 1,
    ('q08', 'doc02_rag_survey_chunk_0000'): 2,
    ('q08', 'doc02_rag_survey_chunk_0005'): 2,
    ('q08', 'doc09_rag_survey_huang_chunk_0002'): 2,
    ('q08', 'doc10_comprehensive_rag_chunk_0004'): 2,
    ('q08', 'doc02_rag_survey_chunk_0001'): 1,
    ('q08', 'doc10_comprehensive_rag_chunk_0000'): 1,
    ('q08', 'doc10_comprehensive_rag_chunk_0006'): 1,
    ('q09', 'doc06_transformer_chunk_0011'): 2,
    ('q09', 'doc06_transformer_chunk_0012'): 1,
    ('q10', 'doc07_lora_chunk_0016'): 2,
    ('q10', 'doc07_lora_chunk_0035'): 2,
    ('q10', 'doc07_lora_chunk_0013'): 1,
    ('q10', 'doc07_lora_chunk_0017'): 1,
    ('q10', 'doc07_lora_chunk_0026'): 1,
    ('q10', 'doc07_lora_chunk_0027'): 1,
    ('q11', 'doc07_lora_chunk_0013'): 2,
    ('q11', 'doc07_lora_chunk_0000'): 1,
    ('q11', 'doc07_lora_chunk_0014'): 1,
    ('q13', 'doc08_flash_attention_chunk_0010'): 2,
    ('q13', 'doc08_flash_attention_chunk_0003'): 1,
    ('q13', 'doc08_flash_attention_chunk_0013'): 1,
    ('q13', 'doc08_flash_attention_chunk_0015'): 1,
    ('q13', 'doc08_flash_attention_chunk_0036'): 1,
    ('q14', 'doc03_dpr_chunk_0000'): 1,
    ('q14', 'doc03_dpr_chunk_0007'): 1,
    ('q14', 'doc03_dpr_chunk_0010'): 1,
    ('q14', 'doc03_dpr_chunk_0016'): 1,
    ('q14', 'doc03_dpr_chunk_0022'): 1,
    ('q15', 'doc04_ragas_chunk_0008'): 2,
    ('q15', 'doc04_ragas_chunk_0000'): 1,
    ('q15', 'doc02_rag_survey_chunk_0033'): 1,
    ('q15', 'doc09_rag_survey_huang_chunk_0036'): 1,
}

pool_df = pd.read_csv(f'{PROC}/qrels_to_label.csv')
pool_df['label'] = [
    PRE_LABELS.get((r.query_id, r.chunk_id), 0)
    for r in pool_df.itertuples()
]
pool_df.to_csv(f'{PROC}/qrels_prelabeled.csv', index=False)
print(f"Saved {len(pool_df)} pre-labelled rows to {PROC}/qrels_prelabeled.csv")
print("\nLabel distribution:", pool_df['label'].value_counts().to_dict())
print("\nLabel-2 chunks per query:")
print(pool_df[pool_df.label == 2].groupby('query_id').size())

Saved 123 pre-labelled rows to /content/drive/MyDrive/rag_term_paper/data/processed/qrels_prelabeled.csv

Label distribution: {1: 51, 0: 46, 2: 26}

Label-2 chunks per query:
query_id
q01    2
q02    4
q03    1
q04    3
q05    1
q06    2
q07    3
q08    4
q09    1
q10    2
q11    1
q13    1
q15    1
dtype: int64


In [ ]:
# Load Verified Labels
lab = pd.read_csv(f'{PROC}/qrels_labeled.csv')
assert lab['label'].isin([0, 1, 2]).all(), 'Unlabelled rows remain!'

qrels = {}
for r in lab.itertuples():
    qrels.setdefault(r.query_id, {})[r.chunk_id] = int(r.label)

print("Relevant chunks (label=2) per query:")
for q, d in sorted(qrels.items()):
    print(f"  {q}: {sum(v == 2 for v in d.values())} relevant")

Relevant chunks (label=2) per query:
  q01: 2 relevant
  q02: 4 relevant
  q03: 1 relevant
  q04: 3 relevant
  q05: 1 relevant
  q06: 2 relevant
  q07: 3 relevant
  q08: 4 relevant
  q09: 1 relevant
  q10: 2 relevant
  q11: 1 relevant
  q12: 0 relevant
  q13: 1 relevant
  q14: 0 relevant
  q15: 1 relevant


In [ ]:
# Layer 2: Retrieval Evaluation vs Human Labels
def dcg(g):
    return sum(x / np.log2(i + 2) for i, x in enumerate(g))

def evaluate(run, qids, ks=(1, 3, 5, 10)):
    rows = []
    for qid in qids:
        rel = qrels[qid]
        good = {c for c, g in rel.items() if g == 2}
        if not good:
            continue
        ranked = run[qid]
        row = {'query_id': qid}
        for k in ks:
            top = set(ranked[:k])
            row[f'Hit@{k}'] = float(bool(good & top))
            row[f'Recall@{k}'] = len(good & top) / len(good)
        row['MRR'] = next((1/(i+1) for i, c in enumerate(ranked)
                           if c in good), 0.0)
        gains = [rel.get(c, 0) for c in ranked[:10]]
        ideal = sorted(rel.values(), reverse=True)[:10]
        row['nDCG@10'] = dcg(gains) / dcg(ideal) if dcg(ideal) else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

ans = [q['query_id'] for q in queries if q['answerable']]
runs = {
    'Dense':      {q: dense_rank(qidx[q], 10) for q in ans},
    'Dense-HNSW': {q: dense_rank_hnsw(qidx[q], 10) for q in ans},
    'BM25':       {q: bm25_rank(queries[qidx[q]]['query'], 10) for q in ans},
    'Hybrid':     {q: hybrid_rank(qidx[q], 10) for q in ans},
}

per_q = {n: evaluate(r, ans) for n, r in runs.items()}
summary = pd.DataFrame({n: d.drop(columns='query_id').mean()
                         for n, d in per_q.items()}).T
summary.round(3).to_csv(f'{RES}/layer2_retrieval_vs_human_qrels.csv')
print("\n=== LAYER 2: RETRIEVAL VS HUMAN LABELS ===")
display(summary.round(3))

# Category breakdown
cat = {q['query_id']: q['category'] for q in queries}
for n, d in per_q.items():
    breakdown = d.assign(cat=d.query_id.map(cat)).groupby('cat')[
        ['Hit@3', 'MRR']].mean().round(2)
    print(f"\n{n}:"); display(breakdown)

# Hybrid vs Dense wins/ties/losses
m = (per_q['Hybrid'].set_index('query_id')['MRR'] -
     per_q['Dense'].set_index('query_id')['MRR'])
print(f"\nHybrid vs Dense MRR: wins={int((m>0).sum())}, "
      f"ties={int((m==0).sum())}, losses={int((m<0).sum())}")


=== LAYER 2: RETRIEVAL VS HUMAN LABELS ===


,Hit@1,Recall@1,Hit@3,Recall@3,Hit@5,Recall@5,Hit@10,Recall@10,MRR,nDCG@10
Dense,0.538,0.333,0.769,0.494,0.923,0.763,0.923,0.865,0.663,0.787
Dense-HNSW,0.538,0.333,0.769,0.494,0.923,0.763,0.923,0.865,0.663,0.787
BM25,0.308,0.179,0.692,0.474,0.846,0.744,0.923,0.788,0.538,0.738
Hybrid,0.538,0.282,0.846,0.590,0.846,0.712,1.000,0.974,0.686,0.832



Dense:


,Hit@3,MRR
cat,,
conceptual,0.88,0.68
keyword_metric,0.60,0.64



Dense-HNSW:


,Hit@3,MRR
cat,,
conceptual,0.88,0.68
keyword_metric,0.60,0.64



BM25:


,Hit@3,MRR
cat,,
conceptual,0.62,0.52
keyword_metric,0.80,0.57



Hybrid:


,Hit@3,MRR
cat,,
conceptual,1.0,0.77
keyword_metric,0.6,0.55



Hybrid vs Dense MRR: wins=5, ties=6, losses=2
